# Homework week7
In this homework, we'll practice streaming with Kafka (Redpanda) and PyFlink.

We use Redpanda, a drop-in replacement for Kafka. It implements the same protocol, so any Kafka client library works with it unchanged.

For this homework we will be using Green Taxi Trip data from October 2025:

green_tripdata_2025-10.parquet
Setup
We'll use the same infrastructure from the workshop.

Follow the setup instructions: build the Docker image, start the services:

cd 07-streaming/workshop/
docker compose build
docker compose up -d
This gives us:

Redpanda (Kafka-compatible broker) on localhost:9092
Flink Job Manager at http://localhost:8081
Flink Task Manager
PostgreSQL on localhost:5432 (user: postgres, password: postgres)
If you previously ran the workshop and have old containers/volumes, do a clean start:

docker compose down -v
docker compose build
docker compose up -d
Note: the container names (like workshop-redpanda-1) assume the directory is called workshop. If you renamed it, adjust accordingly.



## Question 1. Redpanda version
Run rpk version inside the Redpanda container:

docker exec -it workshop-redpanda-1 rpk version
What version of Redpanda are you running?

In [1]:
!docker exec -it week7-redpanda-1 rpk version

rpk version: v25.3.9
Git ref:     836b4a36ef6d5121edbb1e68f0f673c2a8a244e2
Build date:  2026 Feb 26 07 48 21 Thu
OS/Arch:     linux/amd64
Go version:  go1.24.3

Redpanda Cluster
  node-1  v25.3.9 - 836b4a36ef6d5121edbb1e68f0f673c2a8a244e2


## Question 2. Sending data to Redpanda
Create a topic called green-trips:

docker exec -it workshop-redpanda-1 rpk topic create green-trips
Now write a producer to send the green taxi data to this topic.

Read the parquet file and keep only these columns:

- lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, passenger_count, trip_distance, tip_amount, total_amount

Convert each row to a dictionary and send it to the green-trips topic. You'll need to handle the datetime columns - convert them to strings before serializing to JSON.

Measure the time it takes to send the entire dataset and flush:

```python
from time import time

t0 = time()

# send all rows ...

producer.flush()

t1 = time()
print(f"took {(t1 - t0):.2f} seconds")
```

How long did it take to send the data?

- 10 seconds


In [2]:
!docker exec -it week7-redpanda-1 rpk topic create green-trips

TOPIC        STATUS
green-trips  OK


In [3]:
!uv run python src/producers/producer_green.py

Sent 49416 rows
took 37.99 seconds


## Question 3. Consumer - trip distance
Write a Kafka consumer that reads all messages from the green-trips topic (set auto_offset_reset='earliest').

Count how many trips have a trip_distance greater than 5.0 kilometers.

How many trips have trip_distance > 5?


- 8506


In [4]:
!uv run python src/consumers/consumer_green.py

Listening to green-trips...
Processed 6177 messages... Current trip_distance > 5.0 km count: 933
Processed 12354 messages... Current trip_distance > 5.0 km count: 1778
Processed 18531 messages... Current trip_distance > 5.0 km count: 2541
Processed 24708 messages... Current trip_distance > 5.0 km count: 3378
Processed 30885 messages... Current trip_distance > 5.0 km count: 4187
Processed 37062 messages... Current trip_distance > 5.0 km count: 5063
Processed 43239 messages... Current trip_distance > 5.0 km count: 5772
Processed 49416 messages... Current trip_distance > 5.0 km count: 8506
Traceback (most recent call last):
  File "/home/nevin/GitHub/de-zoomcamp/week7/src/consumers/consumer_green.py", line 27, in <module>
    for message in consumer:
                   ^^^^^^^^
  File "/home/nevin/GitHub/de-zoomcamp/week7/.venv/lib/python3.13/site-packages/kafka/consumer/group.py", line 1213, in __next__
    return next(self._iterator)
  File "/home/nevin/GitHub/de-zoomcamp/week7/.venv/li

# Part 2: PyFlink (Questions 4-6)
For the PyFlink questions, you'll adapt the workshop code to work with the green taxi data. The key differences from the workshop:

Topic name: green-trips (instead of rides)
Datetime columns use lpep_ prefix (instead of tpep_)
You'll need to handle timestamps as strings (not epoch milliseconds)
You can convert string timestamps to Flink timestamps in your source DDL:

lpep_pickup_datetime VARCHAR,
event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
Before running the Flink jobs, create the necessary PostgreSQL tables for your results.

Important notes for the Flink jobs:

Place your job files in workshop/src/job/ - this directory is mounted into the Flink containers at /opt/src/job/
Submit jobs with: docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/your_job.py
The green-trips topic has 1 partition, so set parallelism to 1 in your Flink jobs (env.set_parallelism(1)). With higher parallelism, idle consumer subtasks prevent the watermark from advancing.
Flink streaming jobs run continuously. Let the job run for a minute or two until results appear in PostgreSQL, then query the results. You can cancel the job from the Flink UI at http://localhost:8081
If you sent data to the topic multiple times, delete and recreate the topic to avoid duplicates: docker exec -it workshop-redpanda-1 rpk topic delete green-trips

for question 4 , 5 and 6 the tables are created 
```sql
-- For Question 4: 
CREATE TABLE green_trips_window_5m (
    window_start TIMESTAMP,
    PULocationID INTEGER,
    num_trips BIGINT,
    PRIMARY KEY (window_start, PULocationID)
);
-- For Question 5: 
CREATE TABLE session_stats (
     PULocationID INTEGER,
     window_start TIMESTAMP(3),
     window_end TIMESTAMP(3),
     trip_count BIGINT,
     PRIMARY KEY (PULocationID, window_start)
 );

-- For Question 6: 
CREATE TABLE green_trips_tips_hourly (
    window_start TIMESTAMP,
    total_tip DOUBLE PRECISION,
    PRIMARY KEY (window_start)
);

```

## Question 4. Tumbling window - pickup location
Create a Flink job that reads from green-trips and uses a 5-minute tumbling window to count trips per PULocationID.

Write the results to a PostgreSQL table with columns: window_start, PULocationID, num_trips.

After the job processes all data, query the results:

SELECT PULocationID, num_trips
FROM <your_table>
ORDER BY num_trips DESC
LIMIT 3;
Which PULocationID had the most trips in a single 5-minute window?

- 74

```sql
SELECT PULocationID, num_trips 
 FROM green_trips_window_5m 
 ORDER BY num_trips DESC LIMIT 3;
 
+--------------+-----------+
| pulocationid | num_trips |
|--------------+-----------|
| 74           | 15        |
| 74           | 14        |
| 74           | 13        |
+--------------+-----------+
SELECT 3
Time: 0.036s
```

In [7]:
!docker exec -it week7-jobmanager-1 flink run -py /opt/src/job/q5.py

Job has been submitted with JobID 8e7cebd6b547354e338c08bf3dd9a4e6
^C


## Question 5. Session window - longest streak
Create another Flink job that uses a session window with a 5-minute gap on PULocationID, using lpep_pickup_datetime as the event time with a 5-second watermark tolerance.

A session window groups events that arrive within 5 minutes of each other. When there's a gap of more than 5 minutes, the window closes.

Write the results to a PostgreSQL table and find the PULocationID with the longest session (most trips in a single session).

How many trips were in the longest session?

- 81

```sql
SELECT PULocationID, trip_count 
 FROM session_stats 
 ORDER BY trip_count DESC 
 LIMIT 1;
+--------------+------------+
| pulocationid | trip_count |
|--------------+------------|
| 74           | 81         |
+--------------+------------+
SELECT 1
Time: 0.011s
```

In [8]:
!docker exec -it week7-jobmanager-1 flink run -py /opt/src/job/flink_jobs.py

Job has been submitted with JobID b96c3d124ef8c85a14c08f8c7903831d


## Question 6. Tumbling window - largest tip
Create a Flink job that uses a 1-hour tumbling window to compute the total tip_amount per hour (across all locations).

Which hour had the highest total tip amount?

- 2025-10-16 18:00:00


```sql
SELECT window_start, total_tip 
 FROM green_trips_tips_hourly 
 ORDER BY total_tip DESC LIMIT 1;
+---------------------+-------------------+
| window_start        | total_tip         |
|---------------------+-------------------|
| 2025-10-16 18:00:00 | 524.9599999999998 |
+---------------------+-------------------+
SELECT 1
Time: 0.005s
```